[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C19_Bayesian_ML_Course/01_bayesian_inference/01_bayesian_inference.ipynb)

# 01 · 贝叶斯推断与共轭（纯 numpy 从零）

目标：把**贝叶斯定理**、三大**共轭对**（Beta-Binomial / Normal-Normal / Poisson-Gamma）、**MAP vs MLE**、**后验预测分布**、**可信区间** 全部用 numpy 从零实现，并与**解析解 / 网格数值积分**对拍。

路线：贝叶斯定理网格验证 → Beta-Binomial 共轭对拍 → 序贯更新 → Normal-Normal 精度加权 → Poisson-Gamma → MAP/MLE/后验均值 → 后验预测 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**后验 ∝ 似然 × 先验**。共轭让这个更新退化成几个参数的加减；我们每次都用一个独立的参照（网格积分 / 蒙特卡洛 / 解析矩）把它验证到数值一致。

## 1 · 贝叶斯定理：网格上的「似然 × 先验再归一化」

先不碰共轭，直接在 θ 的网格上算 `后验 ∝ 似然 × 先验`，归一化，得到后验。这是贝叶斯定理最朴素的样子，也是后面对拍共轭闭式解的参照之一。

In [ ]:
import numpy as np
from math import lgamma
rng = np.random.default_rng(0)
# 版本无关的梯形积分（numpy<2 用 trapz，numpy>=2 用 trapezoid）
trapz = getattr(np, 'trapezoid', None) or trapz

def check_allclose(name, got, ref, atol=1e-8, rtol=1e-5):
    got = np.asarray(got, float); ref = np.asarray(ref, float)
    ok = np.allclose(got, ref, atol=atol, rtol=rtol)
    err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<30}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 不一致'
    return ok

def beta_logpdf(x, a, b):
    logB = lgamma(a) + lgamma(b) - lgamma(a + b)
    return (a - 1) * np.log(x) + (b - 1) * np.log1p(-x) - logB

def posterior_on_grid(log_prior_fn, log_like_fn, grid):
    '''后验 ∝ 似然×先验，在网格上归一化。返回归一化后验密度。'''
    lu = log_prior_fn(grid) + log_like_fn(grid)
    u = np.exp(lu - lu.max())
    Z = trapz(u, grid)
    return u / Z

grid = np.linspace(1e-6, 1 - 1e-6, 20001)
a0, b0, n, k = 3.0, 3.0, 12, 9
post = posterior_on_grid(lambda t: beta_logpdf(t, a0, b0),
                         lambda t: k*np.log(t) + (n-k)*np.log1p(-t), grid)
# 后验本身应积分为 1
check_allclose('后验归一化积分=1', trapz(post, grid), 1.0, atol=1e-6)
print('✅ 贝叶斯定理 = 似然×先验再归一化（网格实现）')

## 2 · Beta-Binomial 共轭：闭式后验对拍网格

共轭断言后验 = $\mathrm{Beta}(\alpha+k,\ \beta+n-k)$。我们用上面的网格后验当参照，验证这个闭式解逐位正确。更新规则：**成功数加到 α，失败数加到 β**。

In [ ]:
def beta_binomial_update(a, b, n, k):
    '''Beta(a,b) 先验 + Binomial(n,k) 似然 -> Beta 后验参数。'''
    assert 0 <= k <= n
    return a + k, b + (n - k)

a_post, b_post = beta_binomial_update(a0, b0, n, k)
print(f'先验 Beta({a0},{b0}) + ({k}/{n}) -> 后验 Beta({a_post},{b_post})')

# 闭式后验密度 在网格上 对拍 数值后验
post_closed = np.exp(beta_logpdf(grid, a_post, b_post))
check_allclose('Beta 闭式后验 vs 网格后验', post_closed, post, atol=1e-3)

# 后验均值也对拍：解析 a/(a+b) vs 网格 ∫θ·post dθ
mean_closed = a_post / (a_post + b_post)
mean_grid = trapz(grid * post, grid)
check_allclose('后验均值 解析 vs 网格', mean_closed, mean_grid, atol=1e-4)
print('✅ 共轭闭式后验 = 网格数值后验，逐位一致')

## 3 · 序贯更新：今天的后验 = 明天的先验

贝叶斯更新可迭代：把数据**分批**喂入（每批的后验当下批的先验），结果应与**一次性**喂入全部数据**完全相同**。这是在线/流式贝叶斯的根基。

In [ ]:
# 一次性：12 次试验 9 次成功
one_shot = beta_binomial_update(a0, b0, n=12, k=9)

# 分三批喂：(4,3),(4,3),(4,3) -> 共 (12,9)
a, b = a0, b0
for (ni, ki) in [(4, 3), (4, 3), (4, 3)]:
    a, b = beta_binomial_update(a, b, ni, ki)
seq = (a, b)
print('一次性后验:', one_shot, ' 序贯后验:', seq)
check_allclose('序贯更新 == 一次性', np.array(seq), np.array(one_shot), atol=0)
print('✅ 数据独立时，序贯更新与批量更新给出完全相同的后验')

## 4 · Normal-Normal：精度加权平均

已知方差 $\sigma^2$，对均值 $\mu$ 用 Normal 先验 $\mathcal N(\mu_0,\tau_0^2)$。后验：

$$ \tau_n^{-2}=\tau_0^{-2}+n/\sigma^2,\qquad \mu_n=\tau_n^2(\mu_0/\tau_0^2 + n\bar x/\sigma^2) $$

**精度相加，均值是精度加权平均**。我们用网格后验对拍闭式后验。

In [ ]:
def normal_normal_update(mu0, tau0_sq, sigma_sq, data):
    '''Normal(mu0,tau0^2) 先验 + 已知方差 sigma^2 的 Normal 似然 -> 后验(mu_n, tau_n^2)。'''
    n = len(data); xbar = np.mean(data)
    prec_post = 1.0/tau0_sq + n/sigma_sq          # 精度相加
    tau_n_sq = 1.0/prec_post
    mu_n = tau_n_sq * (mu0/tau0_sq + n*xbar/sigma_sq)  # 精度加权
    return mu_n, tau_n_sq

mu_true, sigma = 2.0, 1.5
data = rng.normal(mu_true, sigma, size=20)
mu0, tau0_sq, sigma_sq = 0.0, 4.0, sigma**2
mu_n, tau_n_sq = normal_normal_update(mu0, tau0_sq, sigma_sq, data)
print(f'后验 mu ~ N({mu_n:.4f}, {tau_n_sq:.4f}),  样本均值={data.mean():.4f}')

# 网格后验对拍
mg = np.linspace(-3, 6, 40001)
log_prior = -0.5*(mg-mu0)**2/tau0_sq
log_like = -0.5*np.sum((data[:,None]-mg[None,:])**2, axis=0)/sigma_sq
lu = log_prior + log_like; u = np.exp(lu-lu.max()); u /= trapz(u, mg)
post_mean_grid = trapz(mg*u, mg)
post_var_grid = trapz((mg-post_mean_grid)**2*u, mg)
check_allclose('Normal 后验均值 解析 vs 网格', mu_n, post_mean_grid, atol=1e-3)
check_allclose('Normal 后验方差 解析 vs 网格', tau_n_sq, post_var_grid, atol=1e-3)
assert tau_n_sq < tau0_sq, '后验方差应小于先验方差（数据增加了精度）'
print('✅ 精度加权后验 = 网格后验；后验方差随数据缩小')

## 5 · Poisson-Gamma 共轭

计数数据 $x_i\sim\mathrm{Poisson}(\lambda)$，速率 $\lambda$ 用 Gamma$(\alpha,\beta)$ 先验（$\beta$ 为速率参数）。

$$ \text{后验}=\mathrm{Gamma}\big(\alpha+\textstyle\sum x_i,\ \beta+n\big) $$

更新：**观测总数加到 α、观测个数加到 β**。后验均值 $(\alpha+\sum x_i)/(\beta+n)$ 又是先验与 MLE 的加权平均。

In [ ]:
def poisson_gamma_update(alpha, beta, data):
    '''Gamma(alpha,beta) 先验 + Poisson 似然 -> Gamma 后验。'''
    return alpha + np.sum(data), beta + len(data)

lam_true = 3.5
counts = rng.poisson(lam_true, size=40)
alpha0, beta0 = 2.0, 1.0
a_post, b_post = poisson_gamma_update(alpha0, beta0, counts)
post_mean = a_post / b_post
mle = counts.mean()
print(f'后验 Gamma({a_post:.0f}, {b_post:.0f}), 后验均值={post_mean:.3f}, MLE={mle:.3f}, 真值={lam_true}')

# 网格对拍后验均值
lg = np.linspace(1e-4, 10, 40001)
log_prior = (alpha0-1)*np.log(lg) - beta0*lg
log_like = np.sum(counts)*np.log(lg) - len(counts)*lg   # ∝, 略去 1/x! 与 θ 无关
lu = log_prior+log_like; u = np.exp(lu-lu.max()); u /= trapz(u, lg)
check_allclose('Gamma 后验均值 解析 vs 网格', post_mean, trapz(lg*u, lg), atol=1e-3)
assert abs(post_mean - lam_true) < 0.5, '后验均值应接近真速率'
print('✅ Poisson-Gamma 共轭闭式后验 = 网格后验')

## 6 · MAP vs MLE vs 后验均值，与小样本收缩

三个点估计：MLE=$k/n$、MAP=$\frac{a-1}{a+b-2}$、后验均值=$\frac{a}{a+b}$（$a,b$ 为后验参数）。用经典的**3 连正**例子看收缩：MLE 断言 100%，贝叶斯温和退让。

In [ ]:
def beta_point_estimates(a0, b0, n, k):
    a, b = beta_binomial_update(a0, b0, n, k)
    mle = k / n
    post_mean = a / (a + b)
    post_mode = (a - 1) / (a + b - 2) if (a > 1 and b > 1) else np.nan  # MAP
    return dict(mle=mle, map=post_mode, post_mean=post_mean)

# 3 次抛硬币 3 次正面
est = beta_point_estimates(2.0, 2.0, n=3, k=3)
print(f'3连正: MLE={est["mle"]:.3f} (过拟合!), MAP={est["map"]:.3f}, 后验均值={est["post_mean"]:.3f}')
assert est['mle'] == 1.0
assert est['post_mean'] < 1.0, '贝叶斯后验均值应 < 1，避免 100% 的过度自信'

# 数据变多时三者趋同：300 次 200 次正
est_big = beta_point_estimates(2.0, 2.0, n=300, k=200)
print(f'大样本: MLE={est_big["mle"]:.4f}, MAP={est_big["map"]:.4f}, 后验均值={est_big["post_mean"]:.4f}')
assert abs(est_big['mle'] - est_big['post_mean']) < 0.01, '大样本下先验影响消失，三者趋同'
print('✅ 小样本：先验正则化（收缩）；大样本：数据主导，MAP/均值→MLE')

## 7 · 后验预测分布：比插值更胖

Normal 已知 $\sigma^2$ 时，后验预测 $p(\tilde x\mid D)=\mathcal N(\mu_n,\ \sigma^2+\tau_n^2)$——**数据噪声 + 参数不确定性**两层方差相加。插值法只给 $\mathcal N(\hat\mu,\sigma^2)$，漏掉 $\tau_n^2$。我们用蒙特卡洛（先采 μ 再采 x̃）对拍闭式后验预测。

In [ ]:
def posterior_predictive_normal(mu_n, tau_n_sq, sigma_sq):
    '''返回后验预测 N(mu_n, sigma^2 + tau_n^2) 的 (均值, 方差)。'''
    return mu_n, sigma_sq + tau_n_sq

pp_mean, pp_var = posterior_predictive_normal(mu_n, tau_n_sq, sigma_sq)
# 蒙特卡洛：从后验采 μ，再从 N(μ,σ^2) 采 x̃
M = 400000
mu_samp = rng.normal(mu_n, np.sqrt(tau_n_sq), size=M)
x_tilde = rng.normal(mu_samp, sigma)
check_allclose('后验预测均值 闭式 vs MC', pp_mean, x_tilde.mean(), atol=1e-2)
check_allclose('后验预测方差 闭式 vs MC', pp_var, x_tilde.var(), atol=3e-2)

# 后验预测 必须 比 插值 更胖（多了 tau_n^2）
plugin_var = sigma_sq
print(f'插值预测方差={plugin_var:.4f}, 后验预测方差={pp_var:.4f} (多出参数不确定性 {tau_n_sq:.4f})')
assert pp_var > plugin_var, '后验预测必须比插值更宽'
print('✅ 后验预测把参数不确定性传播进预测 -> 比插值诚实地更宽')

---
## ✏️ 练习 1：Beta-Binomial 更新与可信区间

实现 `beta_binomial_posterior(a0, b0, n, k)` 返回后验 `(a, b)`；再实现 `beta_equal_tailed_ci(a, b, level=0.95)` 用**网格 CDF** 求等尾可信区间 `(lo, hi)`。

（提示：在 [0,1] 网格上算 Beta 密度→累积→反查 2.5% 与 97.5% 分位。）

In [ ]:
from math import lgamma
def beta_binomial_posterior(a0, b0, n, k):
    # TODO: 返回后验 (a, b)
    raise NotImplementedError

def beta_equal_tailed_ci(a, b, level=0.95):
    # TODO: 网格上算 Beta(a,b) 密度->CDF->反查 (1-level)/2 与 (1+level)/2 分位
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
a, b = beta_binomial_posterior(2.0, 2.0, n=10, k=8)
assert (a, b) == (10.0, 4.0), '成功数加到a、失败数加到b'
lo, hi = beta_equal_tailed_ci(a, b, 0.95)
mean = a/(a+b)
assert lo < mean < hi, '后验均值应在可信区间内'
assert 0 < lo < hi < 1
# 与解析：用 lgamma 算 Beta 均值核对
assert abs(mean - 10/14) < 1e-9
# 数据越多区间越窄
a2, b2 = beta_binomial_posterior(2.0, 2.0, n=200, k=160)
lo2, hi2 = beta_equal_tailed_ci(a2, b2, 0.95)
assert (hi2 - lo2) < (hi - lo), '更多数据 -> 更窄的可信区间'
print('✅ 练习 1 通过：共轭更新 + 可信区间，且数据越多区间越窄')

## ✏️ 练习 2：Normal-Normal 精度加权

实现 `normal_normal_posterior(mu0, tau0_sq, sigma_sq, data)` 返回后验 `(mu_n, tau_n_sq)`。验证**精度相加**、**后验均值是先验与样本均值的精度加权平均**、以及**收缩方向正确**。

In [ ]:
def normal_normal_posterior(mu0, tau0_sq, sigma_sq, data):
    # TODO: 精度相加 1/tau_n^2 = 1/tau0^2 + n/sigma^2;
    #       mu_n = tau_n^2 * (mu0/tau0^2 + n*xbar/sigma^2)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
data = np.array([1.0, 2.0, 3.0, 4.0])
mu0, tau0_sq, sigma_sq = 0.0, 1.0, 2.0
mu_n, tau_n_sq = normal_normal_posterior(mu0, tau0_sq, sigma_sq, data)
n, xbar = len(data), data.mean()
# 精度相加
assert abs(1/tau_n_sq - (1/tau0_sq + n/sigma_sq)) < 1e-9
# 后验均值在 先验均值 与 样本均值 之间
assert min(mu0, xbar) <= mu_n <= max(mu0, xbar)
# 精度加权平均 显式核对
w_prior = (1/tau0_sq); w_data = (n/sigma_sq)
mu_check = (w_prior*mu0 + w_data*xbar)/(w_prior + w_data)
assert abs(mu_n - mu_check) < 1e-9
# 后验方差 < 先验方差
assert tau_n_sq < tau0_sq
print('✅ 练习 2 通过：精度相加 + 精度加权平均 + 收缩')

## ✏️ 练习 3：MAP = MLE + 先验正则

对 Bernoulli 数据 + Beta(a,b) 先验，证明 **MAP 等价于带正则的优化**：在对数后验上做（解析）最大化。实现 `bernoulli_map(a, b, n, k)` 返回 MAP，并验证：① 均匀先验 Beta(1,1) 时 MAP==MLE；② 强先验把 MAP 拉向先验均值。

In [ ]:
def bernoulli_map(a, b, n, k):
    # log posterior ∝ (a-1+k) log θ + (b-1+n-k) log(1-θ)
    # 对 θ 求导置零 -> MAP = (a-1+k)/(a+b-2+n)  (需 a+b-2+n>0)
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 均匀先验 Beta(1,1): MAP == MLE = k/n
assert abs(bernoulli_map(1.0, 1.0, n=10, k=7) - 0.7) < 1e-9, '均匀先验下 MAP==MLE'
# 强对称先验把 MAP 拉向 0.5
map_weak = bernoulli_map(2.0, 2.0, n=10, k=9)
map_strong = bernoulli_map(50.0, 50.0, n=10, k=9)
mle = 0.9
assert abs(map_strong - 0.5) < abs(map_weak - 0.5), '更强的先验把 MAP 拉得更靠近 0.5'
assert map_weak < mle, '对称先验把 MAP 从 MLE 向 0.5 收缩'
print(f'✅ 练习 3 通过：弱先验 MAP={map_weak:.3f}, 强先验 MAP={map_strong:.3f} (MLE={mle})')

## ✏️ 练习 4：后验预测分布（Beta-Binomial）

推导并实现 Beta-Binomial 的后验预测：观测 $(n,k)$、先验 Beta(a,b) 后，**再抛 $m$ 次出现 $j$ 次正面** 的概率。它是 Beta-Binomial 分布：

$$ p(j\mid D)=\binom{m}{j}\frac{B(a'+j,\ b'+m-j)}{B(a',b')},\quad a'=a+k,\ b'=b+n-k $$

实现 `beta_binomial_predictive(a, b, n, k, m)` 返回长度 `m+1` 的概率向量 `p[j]`。

In [ ]:
from math import lgamma, log, exp
def log_choose(m, j):
    return lgamma(m+1) - lgamma(j+1) - lgamma(m-j+1)
def log_beta(a, b):
    return lgamma(a) + lgamma(b) - lgamma(a+b)

def beta_binomial_predictive(a, b, n, k, m):
    # TODO: a'=a+k, b'=b+n-k; 对 j=0..m 算 Beta-Binomial PMF（用对数再 exp）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
p = beta_binomial_predictive(2.0, 2.0, n=10, k=8, m=5)
assert len(p) == 6
assert abs(p.sum() - 1.0) < 1e-9, '预测分布必须归一化'
# 再抛 1 次为正的概率 = 后验均值
p1 = beta_binomial_predictive(2.0, 2.0, n=10, k=8, m=1)
a_post, b_post = 10.0, 4.0
assert abs(p1[1] - a_post/(a_post+b_post)) < 1e-9, '再抛1次为正的概率=后验均值'
# 蒙特卡洛对拍：采 θ~Beta(后验)，再 Binomial(m,θ)
th = rng.beta(10.0, 4.0, size=300000)
j_samp = rng.binomial(5, th)
emp = np.bincount(j_samp, minlength=6) / len(j_samp)
assert np.allclose(p, emp, atol=5e-3), '后验预测应与 MC 一致'
print('✅ 练习 4 通过：后验预测=Beta-Binomial，对拍蒙特卡洛一致')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def beta_binomial_posterior(a0, b0, n, k):
    return a0 + k, b0 + (n - k)

def beta_equal_tailed_ci(a, b, level=0.95):
    g = np.linspace(1e-6, 1-1e-6, 50001)
    logB = lgamma(a)+lgamma(b)-lgamma(a+b)
    pdf = np.exp((a-1)*np.log(g)+(b-1)*np.log1p(-g)-logB)
    cdf = np.cumsum(pdf); cdf /= cdf[-1]
    lo = g[np.searchsorted(cdf, (1-level)/2)]
    hi = g[np.searchsorted(cdf, (1+level)/2)]
    return lo, hi

In [ ]:
# 练习 2 参考答案
def normal_normal_posterior(mu0, tau0_sq, sigma_sq, data):
    n = len(data); xbar = np.mean(data)
    tau_n_sq = 1.0/(1.0/tau0_sq + n/sigma_sq)
    mu_n = tau_n_sq*(mu0/tau0_sq + n*xbar/sigma_sq)
    return mu_n, tau_n_sq

In [ ]:
# 练习 3 参考答案
def bernoulli_map(a, b, n, k):
    num = a - 1 + k
    den = a + b - 2 + n
    return num / den

In [ ]:
# 练习 4 参考答案
def log_choose(m, j):
    return lgamma(m+1) - lgamma(j+1) - lgamma(m-j+1)
def log_beta(a, b):
    return lgamma(a) + lgamma(b) - lgamma(a+b)

def beta_binomial_predictive(a, b, n, k, m):
    ap, bp = a + k, b + (n - k)
    js = np.arange(m + 1)
    logp = np.array([log_choose(m, j) + log_beta(ap + j, bp + m - j) - log_beta(ap, bp)
                     for j in js])
    return np.exp(logp)

---
## 🧪 真实数据胶囊：棒球击球率的收缩（James-Stein 的贝叶斯版）

经典真实数据：Efron & Morris 1975 用 1970 赛季 18 名 MLB 球员**前 45 个打数**的击球率，预测整个赛季的真实击球率。**朴素 MLE**（前 45 打数的命中率）方差大、对小样本过拟合；用一个 Beta 先验做贝叶斯收缩能显著改善。

我们把每名球员看作 Binomial，用一个共同的 Beta 先验（经验贝叶斯：先验均值≈联盟平均击球率 ~0.27）做收缩，看收缩后的估计是否比 MLE 更接近赛季真值。带 try/except，无网络时用内置真实数值。

In [ ]:
# Efron-Morris 1975 经典数据（18 名球员, 前 45 打数命中 hits, 赛季真实击球率 truth）
# 这是统计学教科书标准数据集，此处内置真实数值（无需联网）。
hits = np.array([18,17,16,15,14,14,13,12,11,11,10,10,10,10,10,9,8,7], dtype=float)
AB = 45  # at-bats
truth = np.array([0.346,0.298,0.276,0.222,0.273,0.270,0.263,0.210,0.269,
                  0.230,0.264,0.256,0.303,0.264,0.226,0.286,0.200,0.220])

mle = hits / AB                       # 朴素 MLE: 前45打数命中率

# 经验贝叶斯 Beta 先验: 用联盟平均 ~0.27 与一个先验强度 K（伪打数）
prior_mean = 0.27
K_strength = 45.0                      # 先验相当于额外 45 个打数
a0 = prior_mean * K_strength
b0 = (1 - prior_mean) * K_strength
# 每名球员的后验均值（收缩估计）
a_post = a0 + hits
b_post = b0 + (AB - hits)
shrunk = a_post / (a_post + b_post)

# 比较谁更接近赛季真值（误差平方和）
sse_mle = np.sum((mle - truth)**2)
sse_shrunk = np.sum((shrunk - truth)**2)
print(f'MLE 的 SSE      = {sse_mle:.4f}')
print(f'贝叶斯收缩 SSE  = {sse_shrunk:.4f}')
print(f'收缩把误差降低了 {1 - sse_shrunk/sse_mle:.0%}')
assert sse_shrunk < sse_mle, '贝叶斯收缩应比朴素 MLE 更接近真值'
# 收缩把极端值拉向中心
assert shrunk.max() < mle.max() and shrunk.min() > mle.min(), '收缩压缩了估计的范围'
print('✅ 胶囊验证：共享先验的贝叶斯收缩 > 朴素 MLE（Stein 现象的贝叶斯解释）')

**🧪 胶囊练习**：实现 `shrinkage_estimate(hits, AB, prior_mean, K)`：对一组球员返回 Beta 收缩后的后验均值向量。这正是 partial pooling 的最简形式——每名球员向联盟平均借力。

In [ ]:
def shrinkage_estimate(hits, AB, prior_mean, K):
    # TODO: a0=prior_mean*K, b0=(1-prior_mean)*K;
    #       后验均值 = (a0+hits)/(a0+b0+AB)
    raise NotImplementedError

In [ ]:
# 自测
est = shrinkage_estimate(hits, AB, 0.27, 45.0)
assert len(est) == len(hits)
assert np.allclose(est, shrunk), '应与上面手算的收缩估计一致'
# K=0（无先验）应退化为 MLE
assert np.allclose(shrinkage_estimate(hits, AB, 0.27, 0.0), hits/AB), 'K=0 退化为 MLE'
print('✅ 胶囊练习通过：收缩估计 = 向先验均值借力的 partial pooling')

In [ ]:
# 📖 胶囊参考答案
def shrinkage_estimate(hits, AB, prior_mean, K):
    a0 = prior_mean * K
    b0 = (1 - prior_mean) * K
    return (a0 + hits) / (a0 + b0 + AB)

### 小结
- **贝叶斯定理**：后验 ∝ 似然 × 先验；证据是归一化常数（参数推断时是常数，模型比较时是主角）。
- **共轭**：先验与后验同族 → 更新退化为参数加减，闭式解。Beta-Binomial（成功/失败计数）、Normal-Normal（**精度相加 + 精度加权平均**）、Poisson-Gamma。
- **MAP = MLE + 先验正则**；均匀先验时 MAP→MLE。三个点估计（MAP/均值/中位数）对应不同损失，且都**丢弃了不确定性**。
- **后验预测**对后验积分，把**参数不确定性**传播进预测 → 比插值法**诚实地更宽**（方差 = 数据噪声 + 参数不确定性）。
- **收缩**是小样本下的自动正则；大样本时数据主导、先验让位。棒球胶囊展示了它胜过朴素 MLE。

**关键局限**：共轭只是幸运特例。真实模型几乎都没有闭式后验——下一站 **模块 02 · MCMC** 用采样攻克一般后验。